In [15]:
# ============================================================
# 0. 导入依赖
# ============================================================

import pandas as pd                       # 读取 CSV、处理表格数据
import numpy as np                        # 数值计算（AUC、拼接概率等）
from collections import Counter           # 统计字符频次，用于构建词表
from sklearn.model_selection import train_test_split  # 分层划分训练/验证集
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score  # 三个评估指标

import torch                              # PyTorch 主库
import torch.nn as nn                     # 神经网络层定义
import torch.nn.functional as F           # 常用函数：relu、softmax、pooling 等
from torch.utils.data import Dataset, DataLoader  # 自定义数据集与批加载器
import copy                               # 深拷贝模型参数（保存最佳权重）


# ============================================================
# 1. 一些全局配置（超参数与环境设置）
# ============================================================

max_len = 200           # 每条文本最大长度：按字符截断/填充到 200（保证张量形状一致）
min_freq = 2            # 字符在语料中至少出现 2 次才纳入词表（过滤极低频噪声）
max_vocab_size = 20000  # 词表上限（高频字符最多保留 20000 个，控制显存与计算量）

batch_size = 64         # 训练 batch 大小（一次喂给模型 64 条样本）
num_epochs = 5          # 训练轮数（你可以先用 3 跑通，再改为 5 或更多）

# 自动选择设备：如果可用 CUDA，则用 GPU；否则退回 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 数据路径（默认当前目录下）
train_path = "train.csv"
atest_path = "Atest.csv"


# ============================================================
# 2. 构建字符级词表 & 编码函数
#    核心思想：中文短文本用“字符级”是常见做法
# ============================================================

def build_vocab(texts, min_freq=2, max_size=20000):
    """
    根据所有文本构建字符级词表。
    输出 vocab: dict，{token: index}
      - index=0 预留给 <pad>（padding）
      - index=1 预留给 <unk>（未知字符）
    """

    counter = Counter()  # Counter 用于统计每个字符出现次数

    # 遍历所有文本，逐字符统计频次
    for t in texts:
        for ch in str(t):
            counter[ch] += 1

    # counter.most_common(max_size) 返回按频次从高到低的列表
    # 这里同时过滤掉频次 < min_freq 的字符
    most_common = [
        ch for ch, freq in counter.most_common(max_size)
        if freq >= min_freq
    ]

    # 初始化 vocab，先放入两个特殊 token
    vocab = {"<pad>": 0, "<unk>": 1}

    # 按频次顺序把字符加入词表，并为每个字符分配一个新 id
    for ch in most_common:
        if ch not in vocab:
            vocab[ch] = len(vocab)

    return vocab


def encode_text(text, vocab, max_len):
    """
    把一条文本转成“定长”的字符 id 序列（长度=max_len）：
      1) 逐字符查 vocab 得到 id；不在词表则用 <unk>
      2) 如果长度不足 max_len，使用 <pad> 补齐
      3) 如果长度超过 max_len，截断
    """

    ids = []  # 存储当前文本的字符 id 序列

    # 将文本转字符串后逐字符编码
    for ch in str(text):
        # vocab.get(ch, vocab["<unk>"])：若字符不在词表中，用 <unk>
        ids.append(vocab.get(ch, vocab["<unk>"]))

    # padding：不足 max_len 追加 <pad> 的 id（0）
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    else:
        # truncation：超过 max_len 则截断
        ids = ids[:max_len]

    return ids


# ============================================================
# 3. 自定义 Dataset：给 DataLoader 提供按 idx 取样能力
# ============================================================

class TextDataset(Dataset):
    def __init__(self, input_ids, labels=None):
        """
        input_ids: List[List[int]]，每条样本是长度=max_len 的 id 序列
        labels: None 或者 numpy array / list
                - labels=None 表示测试集（无标签）
                - labels 非空表示训练/验证集（有标签）
        """
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self):
        # DataLoader 会用这个决定数据集大小
        return len(self.input_ids)

    def __getitem__(self, idx):
        """
        返回单条样本：
          - 若是训练/验证：返回 (x, y)
          - 若是测试：返回 x
        """
        # x：把该样本的 id 序列转成 LongTensor（Embedding 需要 long）
        x = torch.tensor(self.input_ids[idx], dtype=torch.long)

        if self.labels is None:
            # 测试集没有 y，直接返回 x
            return x

        # y：二分类标签，用 float32（因为后面用 BCEWithLogitsLoss）
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y


# ============================================================
# 4. 模型 1：FastText 风格（Embedding + 平均池化 + FC）
#    适合做快速深度学习 baseline
# ============================================================

class FastTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, padding_idx=0):
        super().__init__()

        # embedding 表：把字符 id 映射为 embed_dim 维向量
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

        # dropout：训练时随机置零一部分特征，防止过拟合
        self.dropout = nn.Dropout(0.5)

        # 全连接：embed_dim -> 1，输出一个 logit（不做 sigmoid）
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, input_ids):
        """
        input_ids: [B, L]，B=batch_size，L=max_len
        返回：logits [B]（每条样本一个 logit）
        """

        # 1) 查 embedding：得到 [B, L, D]
        emb = self.embedding(input_ids)

        # 2) mask：pad 的位置 id=0，非 pad 为 1
        #    (input_ids != 0) 得到 [B, L] 的 bool，再扩一维 -> [B, L, 1]
        mask = (input_ids != 0).unsqueeze(-1)

        # 3) 把 pad 的 embedding 置 0，避免影响后续平均池化
        emb = emb * mask

        # 4) 对时间维求和： [B, L, D] -> [B, D]
        summed = emb.sum(dim=1)

        # 5) 计算有效长度（非 pad 的 token 数）：[B, L, 1] -> [B, 1]
        lengths = mask.sum(dim=1).clamp(min=1)  # clamp 避免除 0

        # 6) 平均池化：得到句向量 [B, D]
        pooled = summed / lengths

        # 7) dropout + fc：得到 [B, 1]
        logits = self.fc(self.dropout(pooled))

        # 8) squeeze：把 [B, 1] 变成 [B]
        return logits.squeeze(-1)


# ============================================================
# 5. 模型 2：TextCNN（Embedding -> 多卷积核 -> max pooling -> FC）
#    通过卷积提取局部 n-gram 特征（类似字 n-gram）
# ============================================================

class TextCNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128,
                 num_filters=100, kernel_sizes=(3, 4, 5),
                 padding_idx=0):
        super().__init__()

        # 字符 embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

        # 多个卷积层（不同 kernel_size 提取不同长度的局部模式）
        # Conv1d 输入形状是 (N, C, L)，所以 channel=C=embed_dim
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim,
                      out_channels=num_filters,
                      kernel_size=k)
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(0.5)

        # 拼接后维度 = num_filters * len(kernel_sizes)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)

    def forward(self, input_ids):
        """
        input_ids: [B, L]
        返回 logits: [B]
        """

        # 1) embedding：[B, L, D]
        emb = self.embedding(input_ids)

        # 2) transpose 以适配 Conv1d： [B, L, D] -> [B, D, L]
        x = emb.transpose(1, 2)

        conv_outs = []  # 存每个卷积核的 pooled 输出

        # 3) 对每个卷积核：
        for conv in self.convs:
            c = conv(x)  # [B, num_filters, L']，L' = L - k + 1
            c = F.relu(c)

            # 4) max pooling：对时间维做全局最大池化
            c = F.max_pool1d(c, kernel_size=c.size(2))  # [B, num_filters, 1]

            # 5) 去掉最后一维：[B, num_filters]
            c = c.squeeze(2)

            conv_outs.append(c)

        # 6) 拼接多个卷积核输出：[B, num_filters * len(kernel_sizes)]
        cat = torch.cat(conv_outs, dim=1)

        # 7) dropout + fc -> [B, 1]
        logits = self.fc(self.dropout(cat))

        # 8) squeeze -> [B]
        return logits.squeeze(-1)


# ============================================================
# 6. 模型 3：BiLSTM（双向 LSTM + 时间维 max pooling + FC）
# ============================================================

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 num_layers=1, padding_idx=0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

        # bidirectional=True：双向 LSTM
        # batch_first=True：输入输出形状用 [B, L, ...]
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)

        self.dropout = nn.Dropout(0.5)

        # 双向输出维度 = 2*hidden_dim
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids):
        # 1) embedding：[B, L, D]
        emb = self.embedding(input_ids)

        # 2) LSTM 输出：[B, L, 2H]
        outputs, _ = self.lstm(emb)

        # 3) 时间维 max pooling：从 [B, L, 2H] 变成 [B, 2H]
        pooled, _ = torch.max(outputs, dim=1)

        # 4) dropout + fc -> [B, 1]
        logits = self.fc(self.dropout(pooled))

        # 5) squeeze -> [B]
        return logits.squeeze(-1)


# ============================================================
# 7. 模型 4：BiGRU + Attention（注意力聚焦关键片段）
# ============================================================

class BiGRUAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 num_layers=1, padding_idx=0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

        # 双向 GRU 输出：[B, L, 2H]
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, bidirectional=True)

        # 注意力网络：先把 2H 映射到 H，再映射到 1 得到每个时间步分数
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids):
        # 1) embedding：[B, L, D]
        emb = self.embedding(input_ids)

        # 2) GRU 输出：[B, L, 2H]
        outputs, _ = self.gru(emb)

        # 3) 计算注意力打分（未归一化）
        #    score: [B, L, H]
        score = torch.tanh(self.attn(outputs))

        # 4) 映射到标量权重：weights: [B, L]
        weights = self.v(score).squeeze(-1)

        # 5) softmax 得到注意力分布：每条样本的 weights 在 L 维上和为 1
        weights = F.softmax(weights, dim=1)

        # 6) 加权求和得到上下文向量 context：[B, 2H]
        #    weights.unsqueeze(1): [B, 1, L]
        #    outputs: [B, L, 2H]
        #    bmm -> [B, 1, 2H] -> squeeze -> [B, 2H]
        context = torch.bmm(weights.unsqueeze(1), outputs).squeeze(1)

        # 7) dropout + fc -> [B, 1]
        logits = self.fc(self.dropout(context))

        # 8) squeeze -> [B]
        return logits.squeeze(-1)


# ============================================================
# 8. 公共训练 + 预测函数（四个模型共用）
# ============================================================

def train_and_predict(model, train_loader, val_loader, test_loader,
                      num_epochs=3, lr=1e-3, model_name="model"):
    """
    训练模型并在验证集评估，然后在测试集输出概率：
    - 损失函数：BCEWithLogitsLoss（内部带 sigmoid，更稳定）
    - 优化器：Adam
    - 每个 epoch 后在验证集计算 Acc/F1/AUC
    - 保存验证 AUC 最优的模型参数（best_state）
    - 最终对测试集输出 prob（sigmoid(logit)）
    """

    # 把模型放到 GPU/CPU
    model.to(device)

    # 二分类损失：输入 logits，标签 float(0/1)
    criterion = nn.BCEWithLogitsLoss()

    # Adam 优化器，lr 默认 1e-3（对从头训练 DL 常用）
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_auc = 0.0          # 记录最佳验证 AUC
    best_state = None       # 保存最佳模型权重（state_dict）

    # ========================================================
    # 训练循环：epoch 级别
    # ========================================================
    for epoch in range(1, num_epochs + 1):

        # ------------------------
        # (1) 训练阶段
        # ------------------------
        model.train()               # 开启训练模式（dropout 生效）
        total_loss = 0.0

        # 遍历训练批次
        for batch in train_loader:
            input_ids, labels = batch

            # 搬到 GPU/CPU
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()   # 清空上一轮梯度

            logits = model(input_ids)          # 前向：输出 [B]
            loss = criterion(logits, labels)   # 计算损失（logits + 标签）

            loss.backward()         # 反向传播计算梯度
            optimizer.step()        # 更新参数

            # 统计总 loss（乘 batch_size，方便后面算平均）
            total_loss += loss.item() * input_ids.size(0)

        # 训练集平均 loss
        avg_train_loss = total_loss / len(train_loader.dataset)

        # ------------------------
        # (2) 验证阶段
        # ------------------------
        model.eval()                # eval 模式（dropout 关闭）
        all_labels = []             # 收集所有 batch 的标签
        all_probs = []              # 收集所有 batch 的预测概率

        with torch.no_grad():       # 验证不需要梯度，节省显存与时间
            for batch in val_loader:
                input_ids, labels = batch

                input_ids = input_ids.to(device)
                labels = labels.to(device)

                logits = model(input_ids)      # [B]
                probs = torch.sigmoid(logits)  # 把 logit 变成概率 [B]，范围 (0,1)

                # 收集到 CPU，准备 numpy 化后算指标
                all_labels.append(labels.cpu())
                all_probs.append(probs.cpu())

        # 拼接为完整验证集数组
        labels_np = torch.cat(all_labels).numpy()   # [N_val]
        probs_np = torch.cat(all_probs).numpy()     # [N_val]

        # 预测类别：默认阈值 0.5（用于 Acc / F1）
        preds_np = (probs_np >= 0.5).astype(int)

        # 计算指标
        acc = accuracy_score(labels_np, preds_np)
        f1 = f1_score(labels_np, preds_np)

        # AUC 需要概率且要求两类样本都存在，否则会报错
        try:
            auc = roc_auc_score(labels_np, probs_np)
        except ValueError:
            auc = float("nan")

        # 打印每轮训练信息（用于日志截图写报告）
        print(f"[{model_name}] Epoch {epoch}/{num_epochs} "
              f"TrainLoss={avg_train_loss:.4f} "
              f"ValAcc={acc:.4f} ValF1={f1:.4f} ValAUC={auc:.4f}")

        # 如果当前 AUC 更好，则保存权重（深拷贝 state_dict）
        if not np.isnan(auc) and auc > best_auc:
            best_auc = auc
            best_state = copy.deepcopy(model.state_dict())

    # 训练结束后，加载最佳权重（验证 AUC 最优）
    if best_state is not None:
        model.load_state_dict(best_state)

    # ========================================================
    # (3) 在测试集上预测概率（用于提交）
    # ========================================================
    model.eval()
    all_test_probs = []

    with torch.no_grad():
        for batch in test_loader:
            # 注意：测试集 DataLoader 返回的是 x（没有 y）
            input_ids = batch.to(device)

            logits = model(input_ids)          # [B]
            probs = torch.sigmoid(logits)      # [B] 正类概率

            # 转回 CPU 并 numpy 化，便于最后拼接
            all_test_probs.append(probs.cpu().numpy())

    # 拼接所有 batch，得到完整测试集概率 [N_test]
    all_test_probs = np.concatenate(all_test_probs, axis=0)

    return all_test_probs


# ============================================================
# 9. 主流程：从读数据 -> 构词表 -> 编码 -> 训练 -> 输出提交文件
# ============================================================

def main():
    # ------------------------
    # (1) 读取数据
    # ------------------------
    train_df = pd.read_csv(train_path)  # 训练集（含 label）
    test_df = pd.read_csv(atest_path)   # 测试集A（不含 label）

    # 取出文本列并转成 list[str]
    train_texts = train_df["text"].astype(str).tolist()
    test_texts = test_df["text"].astype(str).tolist()

    # 取出标签列并转成 float（因为 BCEWithLogitsLoss 期望 float 目标）
    labels = train_df["label"].astype(float).values  # shape: [N_train]

    # ------------------------
    # (2) 划分训练 / 验证集（分层抽样保证类别比例）
    # ------------------------
    train_texts, val_texts, y_train, y_val = train_test_split(
        train_texts,
        labels,
        test_size=0.1,
        random_state=42,
        stratify=labels
    )

    # ------------------------
    # (3) 构建字符级词表
    # ------------------------
    # 这里用 train + val + test 全量文本构词表：
    # 目的是减少测试集中出现 <unk> 的比例，提高泛化
    # 注意：这不使用 label 信息，只是建立“字符集合”，通常不算泄漏
    all_texts_for_vocab = train_texts + val_texts + test_texts

    vocab = build_vocab(
        all_texts_for_vocab,
        min_freq=min_freq,
        max_size=max_vocab_size
    )

    vocab_size = len(vocab)  # 词表大小（包含 <pad> 与 <unk>）
    print("Vocab size:", vocab_size)

    # ------------------------
    # (4) 把文本编码成定长 id 序列
    # ------------------------
    X_train_ids = [encode_text(t, vocab, max_len) for t in train_texts]  # [N_train, max_len]
    X_val_ids   = [encode_text(t, vocab, max_len) for t in val_texts]    # [N_val, max_len]
    X_test_ids  = [encode_text(t, vocab, max_len) for t in test_texts]   # [N_test, max_len]

    # ------------------------
    # (5) 构建 Dataset 与 DataLoader
    # ------------------------
    train_dataset = TextDataset(X_train_ids, y_train)       # 训练集（含 y）
    val_dataset   = TextDataset(X_val_ids, y_val)           # 验证集（含 y）
    test_dataset  = TextDataset(X_test_ids, labels=None)    # 测试集（无 y）

    # shuffle=True：训练时打乱样本顺序，提高训练稳定性
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    # 验证/测试不需要打乱，batch_size 通常可以更大以加速
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size * 2,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size * 2,
        shuffle=False
    )

    # ------------------------
    # (6) 定义四个模型（共享同一 vocab_size）
    # ------------------------
    models = {
        "fasttext_dl":   FastTextClassifier(vocab_size),         # 平均池化
        "textcnn_dl":    TextCNNClassifier(vocab_size),          # 多卷积核 CNN
        "bilstm_dl":     BiLSTMClassifier(vocab_size),           # 双向 LSTM
        "bigru_attn_dl": BiGRUAttentionClassifier(vocab_size),   # 双向 GRU + 注意力
    }

    # ------------------------
    # (7) 依次训练四个模型并生成提交文件
    # ------------------------
    for name, model in models.items():
        print(f"\n==== Training {name} ====")

        # 训练并在测试集输出概率
        test_probs = train_and_predict(
            model,
            train_loader,
            val_loader,
            test_loader,
            num_epochs=num_epochs,   # 使用全局配置
            lr=1e-3,                 # 学习率（从头训练 DL 常用 1e-3，可微调）
            model_name=name
        )

        # 组装提交格式：两列 id + prob
        out_df = pd.DataFrame({
            "id": test_df["id"],
            "prob": test_probs
        })

        # 保存提交文件，文件名包含模型名称，便于后续集成
        out_df.to_csv(
            f"{name}_submission_a.csv",
            index=False,
            encoding="utf-8"
        )

        print(f"Saved {name}_submission_a.csv")


# ============================================================
# 10. 运行主函数
# ============================================================
# 在 Notebook 中取消注释即可开始训练（会跑完四个模型）
# main()


In [16]:
main()

Vocab size: 3748

==== Training fasttext_dl ====
[fasttext_dl] Epoch 1/5 TrainLoss=0.6874 ValAcc=0.6603 ValF1=0.5552 ValAUC=0.7562
[fasttext_dl] Epoch 2/5 TrainLoss=0.6628 ValAcc=0.7092 ValF1=0.6298 ValAUC=0.8223
[fasttext_dl] Epoch 3/5 TrainLoss=0.6339 ValAcc=0.7554 ValF1=0.7115 ValAUC=0.8471
[fasttext_dl] Epoch 4/5 TrainLoss=0.6019 ValAcc=0.7826 ValF1=0.7576 ValAUC=0.8625
[fasttext_dl] Epoch 5/5 TrainLoss=0.5672 ValAcc=0.8043 ValF1=0.7895 ValAUC=0.8764
Saved fasttext_dl_submission_a.csv

==== Training textcnn_dl ====
[textcnn_dl] Epoch 1/5 TrainLoss=0.6264 ValAcc=0.8234 ValF1=0.8285 ValAUC=0.9210
[textcnn_dl] Epoch 2/5 TrainLoss=0.4233 ValAcc=0.8668 ValF1=0.8620 ValAUC=0.9530
[textcnn_dl] Epoch 3/5 TrainLoss=0.3257 ValAcc=0.8886 ValF1=0.8852 ValAUC=0.9670
[textcnn_dl] Epoch 4/5 TrainLoss=0.2624 ValAcc=0.8940 ValF1=0.8870 ValAUC=0.9746
[textcnn_dl] Epoch 5/5 TrainLoss=0.2126 ValAcc=0.9049 ValF1=0.9003 ValAUC=0.9770
Saved textcnn_dl_submission_a.csv

==== Training bilstm_dl ====
[bilst